# Pairwise LLM Evaluation: Replacing Generate-then-Score with Single-Stage Comparison

When you use an LLM to judge generated content (marketing copy, chatbot answers, summaries), the common first approach is **absolute scoring**: ask the model to rate each candidate 1–10. In production this tends to fail in two ways:

1. **Score compression & inconsistency** — most candidates get a 7 or 8, and the same candidate gets different scores across runs, so rankings don't generalize across evaluation rounds.
2. **Unnecessary call layers** — a legacy pattern is *generate feedback → score the feedback*, doubling latency and cost.

**Pairwise comparison** ("which of these two is better?") is a simpler judgment that LLMs make far more consistently. In a production marketing-copy evaluation system, replacing a two-stage generate-then-score pipeline with single-stage pairwise comparison (plus a GPU fix) cut end-to-end runtime from **10+ hours to 15–20 minutes**, and — unlike the legacy metrics — pairwise rankings stayed consistent across PoC rounds.

This notebook shows the pattern:

1. A pairwise judge with structured output
2. Position-bias mitigation by swapping candidate order
3. Round-robin aggregation into win rates
4. **Validating the evaluator itself** against real-world outcomes (agreement, top-k hit rate)

## 1. Setup

In [ ]:
%pip install openai pandas -q

import os
import json
import itertools
import pandas as pd
from openai import OpenAI

client = OpenAI()  # requires OPENAI_API_KEY
MODEL = "gpt-4o-mini" 

## 2. Example data

Candidate push-notification titles for the same campaign, plus (later) real-world open rates so we can validate the evaluator. In a real system, candidates come from your generation pipeline and outcomes from your analytics.

In [ ]:
candidates = {
    "A": "Last chance: 30% off ends tonight",
    "B": "Your cart misses you - and it's 30% cheaper today",
    "C": "30% discount available for a limited period",
    "D": "Tonight only! Extra 30% off your favorites",
}

# Real-world outcomes (e.g., observed open rates) - used ONLY for validating the evaluator
observed_open_rate = {"A": 0.081, "B": 0.114, "C": 0.052, "D": 0.097}

audience = "Price-sensitive shoppers in their 30s who abandoned a cart this week." 

## 3. A pairwise judge with structured output

Two design points:

- **Constrained output** — the judge returns strict JSON (`winner`, `confidence`, `reason`), so downstream aggregation never parses free text.
- **Position-bias mitigation** — LLM judges systematically favor one position (often the first). We ask twice with the order swapped and only count a *decisive* win when both orders agree; disagreements are ties.

In [ ]:
SYSTEM_PROMPT = """You are an expert marketing copy evaluator.
Given an audience description and two push-notification titles,
decide which title is more likely to be opened by that audience.

Respond in JSON: {"winner": "1" or "2", "confidence": 0.0-1.0, "reason": "<one sentence>"}"""

def judge_once(title_1: str, title_2: str) -> dict:
    resp = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Audience: {audience}\n\nTitle 1: {title_1}\nTitle 2: {title_2}"},
        ],
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)

def judge_pair(key_a: str, key_b: str) -> str | None:
    """Judge with both orderings; return winner key, or None on disagreement (tie)."""
    r1 = judge_once(candidates[key_a], candidates[key_b])   # a first
    r2 = judge_once(candidates[key_b], candidates[key_a])   # b first
    w1 = key_a if r1["winner"] == "1" else key_b
    w2 = key_b if r2["winner"] == "1" else key_a
    return w1 if w1 == w2 else None

## 4. Round-robin aggregation

Every candidate meets every other candidate once (both orderings inside `judge_pair`). We aggregate into a **pairwise win rate**: decisive wins / decisive matches.

In [ ]:
wins = {k: 0 for k in candidates}
decisive = {k: 0 for k in candidates}

for a, b in itertools.combinations(candidates, 2):
    w = judge_pair(a, b)
    if w is not None:
        wins[w] += 1
        decisive[a] += 1
        decisive[b] += 1

results = pd.DataFrame({
    "title": candidates,
    "pairwise_win_rate": {k: wins[k] / max(decisive[k], 1) for k in candidates},
}).sort_values("pairwise_win_rate", ascending=False)

results

## 5. Validate the evaluator against reality

An evaluator is only useful if its rankings track real outcomes. Two cheap checks:

- **Rank agreement** — Spearman correlation between the evaluator's ranking and observed open rates.
- **Top-k hit rate** — does the evaluator's top pick land in the observed top-k? In production, top-2/3 hit rate is often the metric that matters: you ship a shortlist, not a single winner.

These same metrics also let you compare *two evaluator pipelines* (e.g., legacy vs. pairwise) on equal footing.

In [ ]:
eval_rank = results.index.tolist()                                   # evaluator's ranking
true_rank = sorted(observed_open_rate, key=observed_open_rate.get, reverse=True)

spearman = pd.Series({k: eval_rank.index(k) for k in candidates}).corr(
    pd.Series({k: true_rank.index(k) for k in candidates}), method="spearman")

top2_hit = eval_rank[0] in true_rank[:2]

print(f"Rank agreement (Spearman): {spearman:.2f}")
print(f"Top-2 hit: {top2_hit}")

## Takeaways

- **Pairwise > absolute scoring** for ranking generated content: more consistent, one fewer call layer, and directly aggregatable.
- **Always debias position** — swap orderings and treat disagreement as a tie.
- **Evaluate your evaluator** — agreement with real outcomes (open rates, CTR, acceptance) is what makes an LLM-judge trustworthy, and it generalizes across evaluation rounds where ad-hoc scoring does not.
- At scale, batch the pairwise calls; with `n` candidates you need `n(n-1)` judgments (both orders) — sampling matchups works well beyond ~20 candidates.

*Based on a production evaluation system for enterprise marketing copy; details generalized. More case studies: [github.com/gamemefy/ml-case-studies](https://github.com/gamemefy/ml-case-studies)*